# 02 - Silver Transform EDA

Phân tích tập trung vào schema typed, event-time alignment, weather hourly join, deduplication invariant và mức độ phủ của các trường thời tiết.

## Input Và Output Paths

Input Bronze:

- `s3a://bronze/yellow_taxi/`
- `s3a://bronze/green_taxi/`
- `s3a://bronze/weather/`

Output Silver:

- `s3a://silver/hourly_weather/`
- `s3a://silver/taxi_weather_trips/`

## Setup Spark Và Utilities

In [1]:
from pathlib import Path
import sys

from pyspark.sql.functions import (
    col,
    count as spark_count,
    date_trunc,
    lit,
    max as spark_max,
    min as spark_min,
    sum as spark_sum,
    when,
)

cwd = Path.cwd().resolve()
if (cwd / "utils").exists():
    notebooks_dir = cwd
elif (cwd / "notebooks" / "utils").exists():
    notebooks_dir = cwd / "notebooks"
else:
    notebooks_dir = cwd

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from utils.spark_session import (
    BRONZE_GREEN_PATH,
    BRONZE_WEATHER_PATH,
    BRONZE_YELLOW_PATH,
    SILVER_TAXI_WEATHER_PATH,
    SILVER_WEATHER_PATH,
    get_spark,
    path_exists,
    safe_display,
    show_schema,
)

spark = get_spark("MetroPulse 02 Silver Transform EDA")
print(f"Spark version: {spark.version}")
print(f"Spark timezone: {spark.conf.get('spark.sql.session.timeZone')}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/metropulse-notebook-ivy-20260526/cache
The jars for the packages stored in: /tmp/metropulse-notebook-ivy-20260526/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-866b40b9-648a-4982-b9a1-378ad5c3f066;1.0
	confs: [default]


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 414ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-866b40b9-648a-4982-b9a1-378ad5c3f066
	confs: [default]


26/05/26 03:26:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.1


Spark timezone: America/New_York


## Phạm Vi Bronze Và Silver Đã Materialize

Các path được đọc bên dưới xác định nguồn raw dùng để tham chiếu và hai output Silver thực sự có mặt trên MinIO: hourly weather và enriched taxi-weather facts. Kết luận chỉ được đưa ra đối với các output đã đọc thành công.

In [2]:
dataset_specs = [
    ("bronze_yellow", BRONZE_YELLOW_PATH, "parquet"),
    ("bronze_green", BRONZE_GREEN_PATH, "parquet"),
    ("bronze_weather", BRONZE_WEATHER_PATH, "parquet"),
    ("silver_hourly_weather", SILVER_WEATHER_PATH, "parquet"),
    ("silver_taxi_weather_trips", SILVER_TAXI_WEATHER_PATH, "parquet"),
]

datasets = {}

for dataset_name, path, fmt in dataset_specs:
    try:
        if not path_exists(spark, path):
            print(f"Cảnh báo: Không tìm thấy path `{dataset_name}` tại {path}. Bỏ qua dataset này.")
            continue
        datasets[dataset_name] = spark.read.format(fmt).load(path)
        print(f"Loaded {dataset_name}: {path}")
    except Exception as exc:
        print(f"Cảnh báo: Không thể đọc `{dataset_name}` tại {path}. Lý do: {exc}")

if not datasets:
    print("Cảnh báo: Chưa đọc được dataset nào. Hãy chạy Bronze/Silver pipeline rồi rerun notebook.")

26/05/26 03:26:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Loaded bronze_yellow: s3a://bronze/yellow_taxi/


Loaded bronze_green: s3a://bronze/green_taxi/


Loaded bronze_weather: s3a://bronze/weather/


Loaded silver_hourly_weather: s3a://silver/hourly_weather/


Loaded silver_taxi_weather_trips: s3a://silver/taxi_weather_trips/


## Silver Schemas

Schema cho thấy `hourly_weather` đã có grain theo giờ, còn `taxi_weather_trips` giữ grain theo chuyến đi sau khi bổ sung weather tại `pickup_hour`.

In [3]:
for dataset_name in ["silver_hourly_weather", "silver_taxi_weather_trips"]:
    df = datasets.get(dataset_name)
    print(f"\n=== {dataset_name} schema ===")
    if df is None:
        print(f"Cảnh báo: `{dataset_name}` không khả dụng, bỏ qua schema.")
    else:
        show_schema(df)
        safe_display(df.limit(5), n=5, truncate=False)


=== silver_hourly_weather schema ===
root
 |-- weather_hour: timestamp (nullable = true)
 |-- weather_timestamp: timestamp (nullable = true)
 |-- source_weather_timestamp: string (nullable = true)
 |-- weather_date: date (nullable = true)
 |-- weather_latitude: double (nullable = true)
 |-- weather_longitude: double (nullable = true)
 |-- weather_location: string (nullable = true)
 |-- temperature_f: double (nullable = true)
 |-- humidity_percent: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- weather_description: string (nullable = true)
 |-- wind_speed_kmh: double (nullable = true)
 |-- wind_direction_deg: double (nullable = true)
 |-- cloud_cover_percent: double (nullable = true)
 |-- weather_source: string (nullable = true)
 |-- silver_processed_timestamp: timestamp (nullable = true)
 |-- weather_year_month: string (nullable = true)



+-------------------+-------------------+------------------------+------------+----------------+-----------------+----------------+-------------+----------------+----------------+------------+-------------------+--------------+------------------+-------------------+--------------+--------------------------+------------------+
|weather_hour       |weather_timestamp  |source_weather_timestamp|weather_date|weather_latitude|weather_longitude|weather_location|temperature_f|humidity_percent|precipitation_mm|weather_code|weather_description|wind_speed_kmh|wind_direction_deg|cloud_cover_percent|weather_source|silver_processed_timestamp|weather_year_month|
+-------------------+-------------------+------------------------+------------+----------------+-----------------+----------------+-------------+----------------+----------------+------------+-------------------+--------------+------------------+-------------------+--------------+--------------------------+------------------+
|2024-01-01 00:0

26/05/26 03:26:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------+---------+-------+-----------------------+--------------------------+-------------------------------+---------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+---------+--------------------+-----------+----------------------+-----------------------+-------------------+-------------------+-------------------+-----------+--------------------------+-------------------+------------------------+----------------+-------------+----------------+----------------+------------+-------------------+--------------+------------------+-------------------+-----------------+---------+
|topic          |partition|offset |kafka_timestamp        |bronze_ingestion_timestamp|source_file                    |vendor_id|passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|

## Bronze Vs Silver Schema Comparison

So với Bronze còn giữ `json_data` và Kafka metadata, Silver cung cấp các cột fact typed như `pickup_datetime`, `pickup_hour`, `trip_distance`, `fare_amount`, `temperature_f` và `precipitation_mm`.

In [4]:
schema_comparison_rows = []
schema_columns = [
    "topic",
    "partition",
    "offset",
    "kafka_timestamp",
    "json_data",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_hour",
    "taxi_type",
    "trip_distance",
    "fare_amount",
    "temperature_f",
    "precipitation_mm",
]

for dataset_name, df in datasets.items():
    existing = set(df.columns)
    for column_name in schema_columns:
        schema_comparison_rows.append((dataset_name, column_name, column_name in existing))

if schema_comparison_rows:
    schema_comparison_df = spark.createDataFrame(
        schema_comparison_rows,
        ["dataset_name", "column_name", "is_present"],
    )
    safe_display(schema_comparison_df, n=len(schema_comparison_rows), truncate=False)
else:
    print("Không có dataset nào để so sánh schema.")

+-------------------------+----------------+----------+
|dataset_name             |column_name     |is_present|
+-------------------------+----------------+----------+
|bronze_yellow            |topic           |true      |
|bronze_yellow            |partition       |true      |
|bronze_yellow            |offset          |true      |
|bronze_yellow            |kafka_timestamp |true      |
|bronze_yellow            |json_data       |true      |
|bronze_yellow            |pickup_datetime |false     |
|bronze_yellow            |dropoff_datetime|false     |
|bronze_yellow            |pickup_hour     |false     |
|bronze_yellow            |taxi_type       |true      |
|bronze_yellow            |trip_distance   |false     |
|bronze_yellow            |fare_amount     |false     |
|bronze_yellow            |temperature_f   |false     |
|bronze_yellow            |precipitation_mm|false     |
|bronze_green             |topic           |true      |
|bronze_green             |partition       |true


[Stage 8:=======================================>                   (2 + 1) / 3]



## Row Counts By Silver Dimensions

Các tổng hợp theo `pickup_year_month`, `taxi_type` và `pickup_date` mô tả phân bố của facts theo thời gian và loại taxi trong khi vẫn giữ tính phân tán của phép tính trên Spark.

In [5]:
taxi_weather_df = datasets.get("silver_taxi_weather_trips")
count_dimensions = ["pickup_year_month", "taxi_type", "pickup_date"]

if taxi_weather_df is None:
    print("Cảnh báo: `silver_taxi_weather_trips` không khả dụng, bỏ qua row count summaries.")
else:
    for dimension in count_dimensions:
        if dimension not in taxi_weather_df.columns:
            print(f"Thiếu column `{dimension}`, bỏ qua count theo dimension này.")
            continue
        print(f"\nRow count by `{dimension}`")
        summary_df = (
            taxi_weather_df.groupBy(dimension)
            .agg(spark_count(lit(1)).alias("row_count"))
            .orderBy(col(dimension))
        )
        safe_display(summary_df, n=100, truncate=False)


Row count by `pickup_year_month`



[Stage 9:===>                                                      (2 + 4) / 34]




[Stage 9:================>                                        (10 + 4) / 34]




[Stage 9:===============================>                         (19 + 4) / 34]




[Stage 9:================================================>        (29 + 4) / 34]



...


[Stage 15:============================>                           (17 + 4) / 34]




[Stage 15:==================================>                     (21 + 4) / 34]




[Stage 15:=========================================>              (25 + 4) / 34]




[Stage 15:===============================================>        (29 + 4) / 34]



+-----------+---------+
|pickup_date|row_count|
+-----------+---------+
|2023-01-01 |1463     |
|2023-01-02 |1564     |
|2023-01-03 |2125     |
|2023-01-04 |2383     |
|2023-01-05 |2417     |
|2023-01-06 |2546     |
|2023-01-07 |2154     |
|2023-01-08 |1659     |
|2023-01-09 |2219     |
|2023-01-10 |2171     |
|2023-01-11 |2487     |
|2023-01-12 |2527     |
|2023-01-13 |2550     |
|2023-01-14 |2169     |
|2023-01-15 |1776     |
|2023-01-16 |1584     |
|2023-01-17 |2246     |
|2023-01-18 |2341     |
|2023-01-19 |2635     |
|2023-01-20 |2469     |
|2023-01-21 |2140     |
|2023-01-22 |1774     |
|2023-01-23 |2392     |
|2023-01-24 |2413     |
|2023-01-25 |2679     |
|2023-01-26 |2632     |
|2023-01-27 |2430     |
|2023-01-28 |2063     |
|2023-01-29 |1676     |
|2023-01-30 |2236     |
|2023-01-31 |2311     |
|2023-02-01 |110331   |
|2023-02-02 |115647   |
|2023-02-03 |117913   |
|2023-02-04 |100084   |
|2023-02-05 |85817    |
|2023-02-06 |93327    |
|2023-02-07 |108545   |
|2023-02-08 |110

## Event-Time Coverage

In [6]:
event_time_columns = ["pickup_datetime", "dropoff_datetime", "weather_timestamp"]

if taxi_weather_df is None:
    print("Cảnh báo: `silver_taxi_weather_trips` không khả dụng, bỏ qua event-time coverage.")
else:
    agg_exprs = []
    for column_name in event_time_columns:
        if column_name in taxi_weather_df.columns:
            agg_exprs.extend([
                spark_min(col(column_name)).alias(f"min_{column_name}"),
                spark_max(col(column_name)).alias(f"max_{column_name}"),
            ])
        else:
            print(f"Thiếu column `{column_name}`, bỏ qua coverage cho column này.")

    if agg_exprs:
        safe_display(taxi_weather_df.agg(*agg_exprs), n=1, truncate=False)


[Stage 18:=====>                                                   (3 + 4) / 34]




[Stage 18:==========>                                              (6 + 4) / 34]




[Stage 18:========================>                               (15 + 4) / 34]




[Stage 18:=============================>                          (18 + 4) / 34]




[Stage 18:====================================>                   (22 + 4) / 34]




[Stage 18:===============================================>        (29 + 4) / 34]




[Stage 18:======================================================> (33 + 1) / 34]



+-------------------+-------------------+--------------------+--------------------+---------------------+---------------------+
|min_pickup_datetime|max_pickup_datetime|min_dropoff_datetime|max_dropoff_datetime|min_weather_timestamp|max_weather_timestamp|
+-------------------+-------------------+--------------------+--------------------+---------------------+---------------------+
|2023-01-01 00:01:31|2024-12-31 23:59:58|2023-01-01 00:13:26 |2025-01-01 22:59:33 |2023-01-01 00:00:00  |2024-12-31 23:00:00  |
+-------------------+-------------------+--------------------+--------------------+---------------------+---------------------+



## Timezone Validation

Transform sử dụng timezone `America/New_York` và tạo `pickup_hour` bằng `date_trunc("hour", pickup_datetime)`. Tôi diễn giải kết quả join theo local hour của NYC, tránh trộn lẫn UTC với thời gian hoạt động taxi tại địa phương.

In [7]:
timezone_columns = ["pickup_datetime", "pickup_hour", "weather_timestamp"]

if taxi_weather_df is None:
    print("Cảnh báo: Không có taxi-weather data để validate timezone sample.")
else:
    available_timezone_columns = [column_name for column_name in timezone_columns if column_name in taxi_weather_df.columns]
    if available_timezone_columns:
        safe_display(taxi_weather_df.select(*available_timezone_columns).limit(20), n=20, truncate=False)
    else:
        print("Không tìm thấy các timezone sample columns cần thiết.")

+-------------------+-------------------+-------------------+
|pickup_datetime    |pickup_hour        |weather_timestamp  |
+-------------------+-------------------+-------------------+
|2023-02-25 21:22:02|2023-02-25 21:00:00|2023-02-25 21:00:00|
|2023-02-13 02:04:57|2023-02-13 02:00:00|2023-02-13 02:00:00|
|2023-02-26 00:29:10|2023-02-26 00:00:00|2023-02-26 00:00:00|
|2023-02-14 07:39:00|2023-02-14 07:00:00|2023-02-14 07:00:00|
|2023-02-22 16:04:36|2023-02-22 16:00:00|2023-02-22 16:00:00|
|2023-02-02 11:54:59|2023-02-02 11:00:00|2023-02-02 11:00:00|
|2023-02-01 12:08:14|2023-02-01 12:00:00|2023-02-01 12:00:00|
|2023-02-25 13:49:55|2023-02-25 13:00:00|2023-02-25 13:00:00|
|2023-02-23 17:02:35|2023-02-23 17:00:00|2023-02-23 17:00:00|
|2023-02-23 18:03:31|2023-02-23 18:00:00|2023-02-23 18:00:00|
|2023-02-19 13:41:28|2023-02-19 13:00:00|2023-02-19 13:00:00|
|2023-02-09 11:06:19|2023-02-09 11:00:00|2023-02-09 11:00:00|
|2023-02-06 06:10:54|2023-02-06 06:00:00|2023-02-06 06:00:00|
|2023-02

## Weather Uniqueness

Kết quả hiện tại ghi nhận `17,542` weather hours phân biệt, `0` giờ bị trùng và tối đa `1` row cho mỗi giờ. Vì weather dimension có grain duy nhất theo local hour, phép join theo giờ không tự làm nhân số dòng taxi.

In [8]:
hourly_weather_df = datasets.get("silver_hourly_weather")

if hourly_weather_df is None:
    print("Cảnh báo: `silver_hourly_weather` không khả dụng, bỏ qua weather uniqueness check.")
else:
    if "weather_hour" in hourly_weather_df.columns:
        weather_hour_df = hourly_weather_df.withColumn("weather_join_hour", col("weather_hour"))
    elif "weather_timestamp" in hourly_weather_df.columns:
        weather_hour_df = hourly_weather_df.withColumn("weather_join_hour", date_trunc("hour", col("weather_timestamp")))
    else:
        weather_hour_df = None
        print("Thiếu `weather_hour` hoặc `weather_timestamp`, không thể kiểm tra weather uniqueness.")

    if weather_hour_df is not None:
        weather_counts_df = (
            weather_hour_df.groupBy("weather_join_hour")
            .agg(spark_count(lit(1)).alias("row_count"))
        )
        duplicate_weather_hours = weather_counts_df.where(col("row_count") > 1)
        duplicate_weather_hour_count = duplicate_weather_hours.count()
        print(f"Weather hours with more than one row: {duplicate_weather_hour_count}")
        if duplicate_weather_hour_count > 0:
            safe_display(duplicate_weather_hours.orderBy(col("row_count").desc()), n=50, truncate=False)
        else:
            print("Không phát hiện weather hour bị duplicate trong Silver hourly_weather.")

Weather hours with more than one row: 0
Không phát hiện weather hour bị duplicate trong Silver hourly_weather.


## Taxi-Weather Join Validation

Output enriched hiện có `60,521,651` taxi-weather rows; các trường `weather_timestamp`, `temperature_f` và `precipitation_mm` đều không có dòng null. Coverage `100.00%` cho thấy toàn bộ trips trong snapshot này tìm được weather hour tương ứng.

In [9]:
weather_fields = ["weather_timestamp", "temperature_f", "precipitation_mm"]

if taxi_weather_df is None:
    print("Cảnh báo: `silver_taxi_weather_trips` không khả dụng, bỏ qua join validation.")
else:
    available_weather_fields = [field for field in weather_fields if field in taxi_weather_df.columns]
    if not available_weather_fields:
        print("Không có weather fields trong taxi-weather data để tính coverage.")
    else:
        total_rows = taxi_weather_df.count()
        metric_exprs = [
            spark_sum(when(col(field).isNull(), 1).otherwise(0)).alias(f"null_{field}")
            for field in available_weather_fields
        ]
        metric_row = taxi_weather_df.agg(*metric_exprs).collect()[0]
        coverage_rows = []
        for field in available_weather_fields:
            null_count = int(metric_row[f"null_{field}"] or 0)
            coverage_ratio = ((total_rows - null_count) / total_rows) if total_rows else 0.0
            coverage_rows.append((field, total_rows, null_count, coverage_ratio))

        coverage_df = spark.createDataFrame(
            coverage_rows,
            ["weather_field", "total_rows", "null_rows", "coverage_ratio"],
        )
        safe_display(coverage_df, n=len(coverage_rows), truncate=False)


[Stage 28:======================================================> (33 + 1) / 34]




[Stage 31:=============>                                           (8 + 4) / 34]




[Stage 31:================================>                       (20 + 4) / 34]




[Stage 31:===============================================>        (29 + 4) / 34]



+-----------------+----------+---------+--------------+
|weather_field    |total_rows|null_rows|coverage_ratio|
+-----------------+----------+---------+--------------+
|weather_timestamp|60521651  |0        |1.0           |
|temperature_f    |60521651  |0        |1.0           |
|precipitation_mm |60521651  |0        |1.0           |
+-----------------+----------+---------+--------------+



## Critical Fact Null Profile

In [10]:
critical_fact_columns = ["pickup_datetime", "pickup_hour", "taxi_type", "pu_location_id"]
if taxi_weather_df is None:
    print("Cảnh báo: Không có taxi-weather output để kiểm critical null invariant.")
else:
    available_critical_columns = [name for name in critical_fact_columns if name in taxi_weather_df.columns]
    missing_critical_columns = sorted(set(critical_fact_columns) - set(available_critical_columns))
    if missing_critical_columns:
        print(f"Thiếu critical columns: {missing_critical_columns}")
    if available_critical_columns:
        critical_nulls_df = taxi_weather_df.agg(*[
            spark_sum(when(col(name).isNull(), 1).otherwise(0)).alias(f"null_{name}")
            for name in available_critical_columns
        ])
        safe_display(critical_nulls_df, n=1, truncate=False)


[Stage 36:==================>                                     (11 + 4) / 34]




[Stage 36:=====================>                                  (13 + 4) / 34]




[Stage 36:============================>                           (17 + 4) / 34]




[Stage 36:=====================================>                  (23 + 4) / 34]




[Stage 36:=========================================>              (25 + 4) / 34]




[Stage 36:===================================================>    (31 + 3) / 34]



+--------------------+----------------+--------------+-------------------+
|null_pickup_datetime|null_pickup_hour|null_taxi_type|null_pu_location_id|
+--------------------+----------------+--------------+-------------------+
|0                   |0               |0             |0                  |
+--------------------+----------------+--------------+-------------------+




[Stage 36:======================================================> (33 + 1) / 34]



## Evidence: Weather Grain Và Row Multiplication Risk

Phép left join dùng khóa `pickup_hour = weather_hour`. Với `duplicate_weather_hours = 0` và `max_rows_per_weather_hour = 1`, mỗi trip có tối đa một weather match; bảng evidence bên dưới là bằng chứng trực tiếp rằng weather side không gây row multiplication trong snapshot này.

In [11]:
if hourly_weather_df is None or taxi_weather_df is None:
    print("Cảnh báo: Cần cả `silver_hourly_weather` và `silver_taxi_weather_trips` để tạo join evidence.")
else:
    join_hour_column = "weather_hour" if "weather_hour" in hourly_weather_df.columns else None
    if join_hour_column is None:
        print("Thiếu `weather_hour`; output không chứng minh được grain dùng trong pipeline join.")
    else:
        weather_grain = hourly_weather_df.groupBy(join_hour_column).agg(spark_count(lit(1)).alias("rows_per_hour"))
        weather_metrics = weather_grain.agg(
            spark_count(lit(1)).alias("weather_hour_count"),
            spark_sum(when(col("rows_per_hour") > 1, 1).otherwise(0)).alias("duplicate_weather_hours"),
            spark_max("rows_per_hour").alias("max_rows_per_weather_hour"),
        ).first()
        silver_fact_rows = taxi_weather_df.count()
        proof_status = "pass" if int(weather_metrics["duplicate_weather_hours"] or 0) == 0 else "fail"
        proof_df = spark.createDataFrame([(
            silver_fact_rows,
            int(weather_metrics["weather_hour_count"] or 0),
            int(weather_metrics["duplicate_weather_hours"] or 0),
            int(weather_metrics["max_rows_per_weather_hour"] or 0),
            proof_status,
        )], ["silver_taxi_weather_rows", "weather_hour_count", "duplicate_weather_hours", "max_rows_per_weather_hour", "weather_join_cardinality_status"])
        safe_display(proof_df, n=1, truncate=False)
        print("PASS nghĩa là weather dimension có quan hệ many-to-one từ trip sang hour; join weather không thể tự tạo thêm trip rows.")

+------------------------+------------------+-----------------------+-------------------------+-------------------------------+
|silver_taxi_weather_rows|weather_hour_count|duplicate_weather_hours|max_rows_per_weather_hour|weather_join_cardinality_status|
+------------------------+------------------+-----------------------+-------------------------+-------------------------------+
|60521651                |17542             |0                      |1                        |pass                           |
+------------------------+------------------+-----------------------+-------------------------+-------------------------------+

PASS nghĩa là weather dimension có quan hệ many-to-one từ trip sang hour; join weather không thể tự tạo thêm trip rows.


## Evidence: Deduplication Output Invariants

In [12]:
def duplicate_group_count(df, key_columns, label):
    missing = [column_name for column_name in key_columns if column_name not in df.columns]
    if missing:
        print(f"Thiếu columns cho `{label}` duplicate check: {missing}")
        return
    duplicates_df = (
        df.groupBy(*key_columns)
        .agg(spark_count(lit(1)).alias("row_count"))
        .where(col("row_count") > 1)
    )
    duplicate_count = duplicates_df.count()
    print(f"{label} duplicate group count = {duplicate_count}")
    if duplicate_count > 0:
        safe_display(duplicates_df.orderBy(col("row_count").desc()), n=20, truncate=False)


if taxi_weather_df is None:
    print("Cảnh báo: Không có taxi-weather data để kiểm tra deduplication.")
else:
    kafka_key = ["topic", "partition", "offset"]
    business_key = [
        "taxi_type",
        "source_file",
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "pu_location_id",
        "do_location_id",
        "trip_distance",
        "fare_amount",
        "total_amount",
    ]
    duplicate_group_count(taxi_weather_df, kafka_key, "Kafka offset")
    duplicate_group_count(taxi_weather_df, business_key, "Business key")


[Stage 50:=====>                                                   (3 + 4) / 34]




[Stage 50:========>                                                (5 + 4) / 34]




[Stage 50:===============>                                         (9 + 4) / 34]




[Stage 50:=======================>                                (14 + 4) / 34]




[Stage 50:============================>                           (17 + 4) / 34]



...


[Stage 58:================>                                       (14 + 4) / 48]




[Stage 58:============================================>           (38 + 4) / 48]




[Stage 58:===============================================>        (41 + 4) / 48]



Business key duplicate group count = 0



[Stage 58:======================================================> (47 + 1) / 48]



## Transformation Explanation

- `from_json` với explicit schemas chuyển raw strings từ Bronze thành dữ liệu typed có thể kiểm tra được, trong khi normalization đưa timestamp khác tên của yellow/green taxi về cùng một fact schema.
- `pickup_hour` theo `America/New_York` cung cấp khóa event-time thống nhất để gắn weather với nhu cầu taxi theo bối cảnh địa phương.
- Weather đã đạt một row mỗi giờ và coverage trên enriched facts đạt `100.00%`, nên weather enrichment của snapshot này vừa đầy đủ vừa không làm phình row count do cardinality phía dimension.
- Broadcast hourly weather là lựa chọn phù hợp về tài nguyên: Spark chuyển dimension nhỏ tới workers thay vì shuffle fact table có `60,521,651` rows.
- Các invariant về critical fields và deduplication chứng minh chất lượng của output hiện có; tác động loại bỏ row trong transform chỉ có thể kết luận bằng một profiling run có cùng input snapshot.